# BUG 01 — Colinealidad perfecta

**Unidad 4.c** · Acompaña a `Clase_05_EleccionBinaria` · Notas: cap. 2

> **Este cuaderno contiene un error deliberado.** No lo corrijas todavía: ejecútalo,
> observa la salida y sigue las tareas del final. Las soluciones están en
> [`SOLUCIONES.md`](SOLUCIONES.md), que conviene no abrir antes de intentarlo.

Este cuaderno **no produce ningún error**. Imprime una tabla de regresión de aspecto
normal. Los coeficientes, sin embargo, no son interpretables.

Estimamos el efecto de la educación superior sobre la participación laboral femenina con
los datos de Mroz (1987), usando un modelo de probabilidad lineal.

In [1]:
import pandas as pd
import statsmodels.api as sm

datos = pd.read_csv("../../Clase_05_EleccionBinaria/Mroz.csv")

# Variable dependiente: participación en la fuerza laboral.
datos["participa"] = (datos["lfp"] == "yes").astype(int)

# Educación superior de la esposa (wc) codificada en dos indicadores.
datos["univ_si"] = (datos["wc"] == "yes").astype(int)
datos["univ_no"] = (datos["wc"] == "no").astype(int)

datos[["lfp", "participa", "wc", "univ_si", "univ_no", "age"]].head()

,lfp,participa,wc,univ_si,univ_no,age
0,yes,1,no,0,1,32
1,yes,1,no,0,1,30
2,yes,1,no,0,1,35
3,yes,1,no,0,1,34
4,yes,1,yes,1,0,31


In [2]:
regresores = sm.add_constant(datos[["univ_si", "univ_no", "age"]])
modelo = sm.OLS(datos["participa"], regresores).fit()

print(modelo.summary().tables[1])
print(f"\nN = {int(modelo.nobs)}   R^2 = {modelo.rsquared:.4f}")

                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.5258      0.064      8.225      0.000       0.400       0.651
univ_si        0.3375      0.038      8.854      0.000       0.263       0.412
univ_no        0.1883      0.037      5.064      0.000       0.115       0.261
age           -0.0044      0.002     -1.989      0.047      -0.009   -5.72e-05

N = 753   R^2 = 0.0248


## La lectura ingenua de la tabla

Todos los coeficientes son «significativos». Leídos sin más, dirían:

In [3]:
b_si = modelo.params["univ_si"]
b_no = modelo.params["univ_no"]

print(f"«Tener universidad sube la participación en {b_si:.3f},»")
print(f"«y NO tenerla la sube en {b_no:.3f}.»")
print()
print("¿Puede ser que AMBAS categorías suban la participación respecto")
print("de la misma base? ¿Cuál es la base?")

«Tener universidad sube la participación en 0.338,»
«y NO tenerla la sube en 0.188.»

¿Puede ser que AMBAS categorías suban la participación respecto
de la misma base? ¿Cuál es la base?


## Un número de la salida que delata el problema

`statsmodels` no invierte $X'X$: usa la **pseudoinversa de Moore-Penrose**. Ante una
matriz singular no lanza excepción — devuelve una de las infinitas soluciones, la de
norma mínima. Los errores estándar que reporta son aritméticamente consistentes con esa
solución arbitraria, y por eso se ven razonables.

El único indicio está en el **número de condición** de la matriz de regresores:

In [4]:
print(f"Número de condición: {modelo.condition_number:.3e}")
print()
print("En una matriz bien condicionada es del orden de 10 a 1000.")
print("Un valor cercano a 1e16 —el recíproco de la precisión de punto")
print("flotante— indica singularidad numérica.")

Número de condición: 2.833e+16

En una matriz bien condicionada es del orden de 10 a 1000.
Un valor cercano a 1e16 —el recíproco de la precisión de punto
flotante— indica singularidad numérica.


## Tareas

1. Explica, con el álgebra del capítulo 2, por qué $X'X$ es singular en este modelo.
   *Pista:* ¿qué vale `univ_si + univ_no` para cada observación, y qué columna de $X$ es
   exactamente ese vector?
2. Pídele a un asistente de IA que explique por qué la tabla no es de fiar. **Antes de
   aceptar su respuesta, verifícala:** ¿qué número de la propia salida la confirma?
3. Corrige la especificación en la celda de abajo. Debe seguir estimando el efecto de la
   educación superior, pero con un modelo identificado.
4. Comprueba que el número de condición baja a un valor razonable.

---
Parte del curso de **Econometría I**, Facultad de Ciencias, UNAM.
Ver el [README de la carpeta](README.md) para las otras actividades de depuración.

In [5]:
# Tu corrección aquí.
#
# Pista: la solución consiste en omitir una categoría, que pasa a ser la base
# de comparación contra la cual se interpretan los coeficientes.